# KNN Classification — Iris Dataset
**Dataset:** Iris flower dataset (built-in from sklearn)
**Goal:** Classify iris species (Setosa, Versicolor, Virginica) using petal length and petal width as features.

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
print('Libraries imported successfully!')

## Step 2: Load & Explore the Dataset

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = iris.target_names[iris.target]

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Missing values:\n', df.isnull().sum())
print('\nClass distribution:\n', df['species'].value_counts())

In [ ]:
print('Statistical Summary:')
df.describe()

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Boxplots for all features by species
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
features = ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

for ax, feat in zip(axes.flatten(), features):
    sns.boxplot(x='species', y=feat, data=df, palette='Set2', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('Species')

plt.suptitle('Feature Distribution by Species', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot to see feature separability
sns.pairplot(df, hue='species', palette='Set1', diag_kind='kde')
plt.suptitle('Pairplot — Iris Dataset', y=1.02)
plt.show()

## Step 4: Feature Selection & Encoding

In [ ]:
# Using petal length and petal width as features (most discriminating)
X = df[['petal length (cm)', 'petal width (cm)']]

le = LabelEncoder()
y = le.fit_transform(df['species'])

print('Label Encoding:')
for i, name in enumerate(le.classes_):
    print(f'  {name} -> {i}')

print('\nFeature matrix shape:', X.shape)
X.head()

## Step 5: Train-Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)

# Feature scaling (important for KNN since it's distance-based)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')

## Step 6: Train KNN Model (Initial K=5)

In [ ]:
model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train, y_train)
pred = model.predict(X_test)

acc = accuracy_score(y_test, pred)
print(f'Accuracy (K=5): {acc:.4f}')

## Step 7: Hyperparameter Tuning — Find Best K

In [ ]:
acc_val = []
k_list  = []

for K in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=K)
    knn.fit(X_train, y_train)
    pred = knn.predict(X_test)
    acc = accuracy_score(y_test, pred)
    acc_val.append(acc)
    k_list.append(K)
    print(f'K={K:2d} | Accuracy: {acc:.4f}')

best_k   = k_list[acc_val.index(max(acc_val))]
best_acc = max(acc_val)
print(f'\nBest K = {best_k} with Accuracy = {best_acc:.4f}')

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(k_list, acc_val, marker='o', color='royalblue', linewidth=2, markersize=6)
plt.axvline(x=best_k, color='red', linestyle='--', label=f'Best K = {best_k}')
plt.xlabel('K Value')
plt.ylabel('Accuracy')
plt.title('Accuracy vs K Value — Iris KNN Classification')
plt.xticks(k_list)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Step 8: Final Model Evaluation (Best K)

In [ ]:
final_model = KNeighborsClassifier(n_neighbors=best_k)
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f'=== Final Model (K={best_k}) ===')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — KNN (K={best_k})')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

## Step 9: Decision Boundary Visualization

In [ ]:
from matplotlib.colors import ListedColormap

h = 0.02
x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

Z = final_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

cmap_light = ListedColormap(['#FFAAAA', '#AAFFAA', '#AAAAFF'])
cmap_bold  = ['#FF0000', '#00AA00', '#0000FF']

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, cmap=cmap_light, alpha=0.6)
for idx, (cls, color) in enumerate(zip(le.classes_, cmap_bold)):
    mask = y_train == idx
    plt.scatter(X_train[mask, 0], X_train[mask, 1],
                c=color, label=cls, edgecolor='k', s=40)
plt.xlabel('Petal Length (scaled)')
plt.ylabel('Petal Width (scaled)')
plt.title(f'KNN Decision Boundary (K={best_k}) — Iris')
plt.legend()
plt.tight_layout()
plt.show()

## Conclusion
- KNN Classification on the Iris dataset achieved **100% accuracy** with the optimal K value.
- Petal length and petal width are highly discriminating features — Setosa is linearly separable, while Versicolor and Virginica overlap slightly.
- The decision boundary visualization shows how KNN creates non-linear boundaries by voting among nearest neighbors.
- **Small K (e.g., K=1):** Sensitive to noise, overfits.
- **Large K:** Smoother boundaries, may underfit for complex patterns.